# R2 — Entropy-regularization λ sweep for adaptive gating

This notebook performs a **validation-only λ sweep** for the dual RF/NN adaptive gates.

## Goal

Test whether another entropy coefficient can outperform the fixed **2:1 RF:NN** baseline, while keeping the already-trained RF and NN experts fixed.

For each target independently:

1. Keep the previously selected gate architecture/hyperparameters fixed.
2. Train the adaptive gate for each λ value.
3. Select the best λ using **validation MSE only**.
4. Compare validation performance against the static 2:1 baseline.
5. Evaluate **only the selected λ** on the held-out test set.

The test set is not used to choose λ.\n\n**Scaling convention:** predictions and targets remain in the exported scaler's target space during training. Metrics are computed in original UPDRS units by inverse-transforming both ground truth and predictions exactly once.

### λ values

`[0, 0.0001, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.2, 0.5]`

The proposed paper model currently uses λ=0.05.


In [ ]:
import os
import copy
import json
import random
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

os.makedirs("../results/r2_lambda_sweep", exist_ok=True)


In [ ]:
# ------------------------------------------------------------
# Load the exact processed splits used by the current model
# ------------------------------------------------------------

X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")

y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

y_scaler = joblib.load("../data/processed/y_scaler.pkl")

# ------------------------------------------------------------
# Load the already-trained RF and NN experts
# ------------------------------------------------------------

with open("../models/best_nn_params.json", "r") as f:
    nn_params = json.load(f)

class FinalParkinsonNN(nn.Module):
    def __init__(self, params, input_dim=18):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, params["units_l1"]), nn.ReLU(),
            nn.Linear(params["units_l1"], params["units_l2"]), nn.ReLU(),
            nn.Dropout(params["dropout"]),
            nn.Linear(params["units_l2"], params["units_l3"]), nn.ReLU(),
            nn.Linear(params["units_l3"], params["units_l4"]), nn.ReLU(),
            nn.Linear(params["units_l4"], 2)
        )

    def forward(self, x):
        return self.net(x)

nn_model = FinalParkinsonNN(nn_params).to(DEVICE)
nn_model.load_state_dict(torch.load("../models/best_nn_model.pt", map_location=DEVICE))
nn_model.eval()

rf_expert = joblib.load("../models/rf_expert.pkl")

# ------------------------------------------------------------
# Expert predictions in the existing scaled target space
# ------------------------------------------------------------

def get_nn_preds(X):
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return nn_model(Xt).cpu().numpy()

nn_train_preds = get_nn_preds(X_train)
nn_val_preds = get_nn_preds(X_val)
nn_test_preds = get_nn_preds(X_test)

rf_train_preds = rf_expert.predict(X_train)
rf_val_preds = rf_expert.predict(X_val)
rf_test_preds = rf_expert.predict(X_test)

gate_train = np.concatenate([X_train, rf_train_preds, nn_train_preds], axis=1)
gate_val = np.concatenate([X_val, rf_val_preds, nn_val_preds], axis=1)
gate_test = np.concatenate([X_test, rf_test_preds, nn_test_preds], axis=1)

print("Gate input shape:", gate_train.shape)


In [ ]:

# ------------------------------------------------------------
# IMPORTANT SCALE SANITY CHECK
# ------------------------------------------------------------
# y_train/y_val/y_test are expected to be in the SAME scaled target
# space used to train the RF/NN/gates.
#
# We evaluate predictions in original UPDRS units by inverse-transforming
# BOTH y_true and predictions exactly once.
#
# Check that the scaler is a fitted two-target scaler.
print("y_scaler:", type(y_scaler).__name__)
print("Scaler center shape:", getattr(y_scaler, "mean_", np.array([])).shape)
print("Scaler scale shape:", getattr(y_scaler, "scale_", np.array([])).shape)

# Quick range check after inverse transform.
for name, arr in [
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test),
]:
    original = y_scaler.inverse_transform(arr)
    print(
        f"{name}: scaled range=({arr.min():.4f}, {arr.max():.4f}), "
        f"original range=({original.min():.4f}, {original.max():.4f})"
    )


In [ ]:
class GateDataset(Dataset):
    def __init__(self, X, y, rf_preds, nn_preds):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.rf_preds = torch.tensor(rf_preds, dtype=torch.float32)
        self.nn_preds = torch.tensor(nn_preds, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx],
            self.rf_preds[idx],
            self.nn_preds[idx],
        )


class AdaptiveGate(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate, temperature):
        super().__init__()
        self.temperature = temperature
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        logits = self.net(x)
        return torch.softmax(logits / self.temperature, dim=1)


## Fixed gate configurations

These are the architecture/training settings already selected for the proposed model. **Only λ changes in the sweep.**


In [ ]:
best_motor_gate_params = {
    "hidden_dim": 8,
    "learning_rate": 0.0019122748228268711,
    "dropout_rate": 0.04470145327296845,
    "temperature": 1.0,
    "batch_size": 64,
}

best_total_gate_params = {
    "hidden_dim": 32,
    "learning_rate": 0.0018123769960513428,
    "dropout_rate": 0.07826676875662475,
    "temperature": 1.0,
    "batch_size": 128,
}

# Small, deliberately local sweep around the current λ=0.05.
LAMBDA_VALUES = [
    0.0,
    0.0001,
    0.0005,
    0.001,
    0.0025,
    0.005,
    0.01,
    0.025,
    0.05,
    0.1,
    0.2,
    0.5,
]

print("Lambda sweep:", LAMBDA_VALUES)


In [ ]:
def make_gate(params):
    return AdaptiveGate(
        input_dim=gate_train.shape[1],
        hidden_dim=params["hidden_dim"],
        dropout_rate=params["dropout_rate"],
        temperature=params["temperature"],
    ).to(DEVICE)


def train_one_lambda(
    target_idx,
    lambda_value,
    gate_params,
    max_epochs=200,
    patience=20,
):
    # Fresh deterministic seed for reproducibility.
    # Each λ gets the same initial parameter seed within each target.
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = make_gate(gate_params)

    optimizer = optim.Adam(
        model.parameters(),
        lr=gate_params["learning_rate"],
    )
    mse_loss = nn.MSELoss()

    train_dataset = GateDataset(
        gate_train,
        y_train,
        rf_train_preds,
        nn_train_preds,
    )
    val_dataset = GateDataset(
        gate_val,
        y_val,
        rf_val_preds,
        nn_val_preds,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=gate_params["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=gate_params["batch_size"],
        shuffle=False,
    )

    best_val_mse = np.inf
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()

        for Xb, yb, rfb, nnb in train_loader:
            Xb = Xb.to(DEVICE)
            yb = yb.to(DEVICE)
            rfb = rfb.to(DEVICE)
            nnb = nnb.to(DEVICE)

            optimizer.zero_grad()

            weights = model(Xb)
            pred = (
                weights[:, 0] * rfb[:, target_idx]
                + weights[:, 1] * nnb[:, target_idx]
            )

            pred_loss = mse_loss(pred, yb[:, target_idx])

            entropy = -torch.mean(
                torch.sum(
                    weights * torch.log(weights + 1e-8),
                    dim=1,
                )
            )

            # Entropy is used only as a training regularizer.
            loss = pred_loss - lambda_value * entropy
            loss.backward()
            optimizer.step()

        # IMPORTANT:
        # Early stopping / λ selection uses validation PREDICTION MSE,
        # not regularized validation loss.
        model.eval()

        val_preds = []
        val_true = []
        val_weights = []

        with torch.no_grad():
            for Xb, yb, rfb, nnb in val_loader:
                Xb = Xb.to(DEVICE)
                rfb = rfb.to(DEVICE)
                nnb = nnb.to(DEVICE)

                weights = model(Xb)
                pred = (
                    weights[:, 0] * rfb[:, target_idx]
                    + weights[:, 1] * nnb[:, target_idx]
                )

                val_preds.append(pred.cpu().numpy())
                val_true.append(yb[:, target_idx].numpy())
                val_weights.append(weights.cpu().numpy())

        val_preds = np.concatenate(val_preds)
        val_true = np.concatenate(val_true)
        val_weights = np.concatenate(val_weights)

        val_mse = mean_squared_error(val_true, val_preds)

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        val_tensor = torch.tensor(gate_val, dtype=torch.float32, device=DEVICE)
        final_weights = model(val_tensor).cpu().numpy()

    val_pred_scaled = (
        final_weights[:, 0] * (rf_val_preds[:, target_idx])
        + final_weights[:, 1] * (nn_val_preds[:, target_idx])
    )

    return {
        "model": model,
        "weights": final_weights,
        "pred_scaled": val_pred_scaled,
        "best_val_mse_scaled": mean_squared_error(
            y_val[:, target_idx], val_pred_scaled
        ),
        "best_epoch": best_epoch,
    }


In [ ]:
def evaluate_scaled_prediction(y_true_scaled, pred_scaled, target_idx):
    # Inverse-transform two-target arrays while preserving the other target.
    pred_2d = np.zeros((len(pred_scaled), 2), dtype=float)
    true_2d = np.zeros((len(y_true_scaled), 2), dtype=float)

    pred_2d[:, target_idx] = pred_scaled
    true_2d[:, target_idx] = y_true_scaled[:, target_idx]

    pred_orig = y_scaler.inverse_transform(pred_2d)[:, target_idx]
    true_orig = y_scaler.inverse_transform(true_2d)[:, target_idx]

    return {
        "RMSE": np.sqrt(mean_squared_error(true_orig, pred_orig)),
        "MAE": mean_absolute_error(true_orig, pred_orig),
        "R2": r2_score(true_orig, pred_orig),
        "true_orig": true_orig,
        "pred_orig": pred_orig,
    }


def routing_stats(weights):
    rf_w = weights[:, 0]
    nn_w = weights[:, 1]
    entropy = -np.sum(weights * np.log(weights + 1e-8), axis=1)

    return {
        "mean_rf_weight": float(np.mean(rf_w)),
        "mean_nn_weight": float(np.mean(nn_w)),
        "median_rf_weight": float(np.median(rf_w)),
        "median_nn_weight": float(np.median(nn_w)),
        "rf_dominant_pct": float(np.mean(rf_w > nn_w) * 100.0),
        "nn_dominant_pct": float(np.mean(nn_w >= rf_w) * 100.0),
        "mean_entropy": float(np.mean(entropy)),
    }


## Validation sweep

This is the key selection stage. A λ is considered better when its **validation RMSE / MAE / R²** is better. The test set is still untouched.


In [ ]:
results = []
models_by_target_lambda = {}

target_specs = {
    "Motor-UPDRS": (0, best_motor_gate_params),
    "Total-UPDRS": (1, best_total_gate_params),
}

for target_name, (target_idx, gate_params) in target_specs.items():
    print(f"\n{'='*70}")
    print(target_name)
    print(f"{'='*70}")

    models_by_target_lambda[target_name] = {}

    for lam in LAMBDA_VALUES:
        print(f"Training λ={lam} ...", end=" ", flush=True)

        result = train_one_lambda(
            target_idx=target_idx,
            lambda_value=lam,
            gate_params=gate_params,
        )

        eval_result = evaluate_scaled_prediction(
            y_val,
            result["pred_scaled"],
            target_idx,
        )

        route = routing_stats(result["weights"])

        row = {
            "Target": target_name,
            "Lambda": lam,
            "Val_RMSE": eval_result["RMSE"],
            "Val_MAE": eval_result["MAE"],
            "Val_R2": eval_result["R2"],
            "Best_Epoch": result["best_epoch"],
            **route,
        }

        results.append(row)

        models_by_target_lambda[target_name][lam] = result

        print(
            f"RMSE={eval_result['RMSE']:.4f}, "
            f"MAE={eval_result['MAE']:.4f}, "
            f"R2={eval_result['R2']:.4f}, "
            f"RF={route['mean_rf_weight']:.3f}"
        )

sweep_df = pd.DataFrame(results)
sweep_df.to_csv(
    "../results/r2_lambda_sweep/validation_lambda_sweep.csv",
    index=False,
)

display(sweep_df)


In [ ]:
# ------------------------------------------------------------
# Static 2:1 RF:NN validation baseline
# ------------------------------------------------------------

static_motor_scaled = (
    (2/3) * rf_val_preds[:, 0]
    + (1/3) * nn_val_preds[:, 0]
)

static_total_scaled = (
    (2/3) * rf_val_preds[:, 1]
    + (1/3) * nn_val_preds[:, 1]
)

static_rows = []

for target_name, target_idx, pred_scaled in [
    ("Motor-UPDRS", 0, static_motor_scaled),
    ("Total-UPDRS", 1, static_total_scaled),
]:
    ev = evaluate_scaled_prediction(y_val, pred_scaled, target_idx)
    static_rows.append({
        "Target": target_name,
        "Lambda": "static_2_to_1",
        "Val_RMSE": ev["RMSE"],
        "Val_MAE": ev["MAE"],
        "Val_R2": ev["R2"],
        "mean_rf_weight": 2/3,
        "mean_nn_weight": 1/3,
        "rf_dominant_pct": 100.0,
        "nn_dominant_pct": 0.0,
        "mean_entropy": float(
            -(2/3)*np.log(2/3) -(1/3)*np.log(1/3)
        ),
    })

static_df = pd.DataFrame(static_rows)
static_df.to_csv(
    "../results/r2_lambda_sweep/validation_static_2_to_1.csv",
    index=False,
)

print("Static 2:1 validation baseline:")
display(static_df)


In [ ]:
# ------------------------------------------------------------
# Select the best λ on VALIDATION MSE/RMSE for each target
# ------------------------------------------------------------

best_rows = []

for target_name in target_specs:
    target_df = sweep_df[sweep_df["Target"] == target_name].copy()
    best = target_df.sort_values(
        ["Val_RMSE", "Val_MAE"],
        ascending=[True, True],
    ).iloc[0]

    static_row = static_df[static_df["Target"] == target_name].iloc[0]

    best_rows.append({
        "Target": target_name,
        "Selected_Lambda": float(best["Lambda"]),
        "Best_Val_RMSE": float(best["Val_RMSE"]),
        "Best_Val_MAE": float(best["Val_MAE"]),
        "Best_Val_R2": float(best["Val_R2"]),
        "Static_2_to_1_Val_RMSE": float(static_row["Val_RMSE"]),
        "Static_2_to_1_Val_MAE": float(static_row["Val_MAE"]),
        "Static_2_to_1_Val_R2": float(static_row["Val_R2"]),
        "RMSE_Improvement_vs_Static": float(
            static_row["Val_RMSE"] - best["Val_RMSE"]
        ),
    })

best_lambda_df = pd.DataFrame(best_rows)
best_lambda_df.to_csv(
    "../results/r2_lambda_sweep/best_lambda_validation.csv",
    index=False,
)

display(best_lambda_df)


## Test evaluation of the validation-selected λ

Only the λ selected above is evaluated here.

**Do not select a different λ after seeing these test results.**


In [ ]:
test_rows = []
selected_artifacts = {}

for _, row in best_lambda_df.iterrows():
    target_name = row["Target"]
    target_idx = target_specs[target_name][0]
    selected_lambda = float(row["Selected_Lambda"])

    result = models_by_target_lambda[target_name][selected_lambda]

    model = result["model"]
    model.eval()

    test_tensor = torch.tensor(
        gate_test,
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.no_grad():
        test_weights = model(test_tensor).cpu().numpy()

    test_pred_scaled = (
        test_weights[:, 0] * rf_test_preds[:, target_idx]
        + test_weights[:, 1] * nn_test_preds[:, target_idx]
    )

    ev = evaluate_scaled_prediction(
        y_test,
        test_pred_scaled,
        target_idx,
    )

    route = routing_stats(test_weights)

    # Static 2:1 test baseline
    static_test_scaled = (
        (2/3) * rf_test_preds[:, target_idx]
        + (1/3) * nn_test_preds[:, target_idx]
    )
    static_ev = evaluate_scaled_prediction(
        y_test,
        static_test_scaled,
        target_idx,
    )

    # Existing proposed λ=0.05 exported gate
    proposed_path = (
        "../models/motor_gate.pt"
        if target_idx == 0
        else "../models/total_gate.pt"
    )

    proposed_gate = make_gate(target_specs[target_name][1])
    proposed_gate.load_state_dict(
        torch.load(proposed_path, map_location=DEVICE)
    )
    proposed_gate.eval()

    with torch.no_grad():
        proposed_weights = proposed_gate(test_tensor).cpu().numpy()

    proposed_test_scaled = (
        proposed_weights[:, 0] * rf_test_preds[:, target_idx]
        + proposed_weights[:, 1] * nn_test_preds[:, target_idx]
    )

    proposed_ev = evaluate_scaled_prediction(
        y_test,
        proposed_test_scaled,
        target_idx,
    )

    row_out = {
        "Target": target_name,
        "Selected_Lambda": selected_lambda,

        "Selected_Test_RMSE": ev["RMSE"],
        "Selected_Test_MAE": ev["MAE"],
        "Selected_Test_R2": ev["R2"],

        "Static_2_to_1_Test_RMSE": static_ev["RMSE"],
        "Static_2_to_1_Test_MAE": static_ev["MAE"],
        "Static_2_to_1_Test_R2": static_ev["R2"],

        "Proposed_0.05_Test_RMSE": proposed_ev["RMSE"],
        "Proposed_0.05_Test_MAE": proposed_ev["MAE"],
        "Proposed_0.05_Test_R2": proposed_ev["R2"],

        "Selected_vs_Static_RMSE_Gain": static_ev["RMSE"] - ev["RMSE"],
        "Selected_vs_Static_R2_Gain": ev["R2"] - static_ev["R2"],

        **{f"Selected_{k}": v for k, v in route.items()},
    }

    test_rows.append(row_out)

    selected_artifacts[target_name] = {
        "lambda": selected_lambda,
        "weights_test": test_weights,
        "pred_scaled_test": test_pred_scaled,
        "model": model,
        "routing": route,
    }

test_df = pd.DataFrame(test_rows)
test_df.to_csv(
    "../results/r2_lambda_sweep/selected_lambda_test_results.csv",
    index=False,
)

display(test_df)


In [ ]:
# ------------------------------------------------------------
# Save the validation-selected gate checkpoints
# ------------------------------------------------------------

for target_name, artifact in selected_artifacts.items():
    suffix = "motor" if target_name == "Motor-UPDRS" else "total"
    path = f"../results/r2_lambda_sweep/{suffix}_gate_selected_lambda.pt"

    torch.save(
        artifact["model"].state_dict(),
        path,
    )

selected_params = {
    row["Target"]: {
        "selected_lambda": float(row["Selected_Lambda"])
    }
    for _, row in best_lambda_df.iterrows()
}

with open(
    "../results/r2_lambda_sweep/selected_lambda.json",
    "w",
) as f:
    json.dump(selected_params, f, indent=4)

print(json.dumps(selected_params, indent=4))


## Routing interpretation

This helps determine whether a λ that improves validation performance also avoids complete RF collapse.


In [ ]:
routing_summary = []

for target_name in target_specs:
    target_df = sweep_df[sweep_df["Target"] == target_name].copy()

    cols = [
        "Lambda",
        "Val_RMSE",
        "Val_MAE",
        "Val_R2",
        "mean_rf_weight",
        "mean_nn_weight",
        "rf_dominant_pct",
        "nn_dominant_pct",
        "mean_entropy",
    ]

    print(f"\n{target_name}")
    display(
        target_df.sort_values("Val_RMSE")[cols].reset_index(drop=True)
    )

    target_df[cols].to_csv(
        f"../results/r2_lambda_sweep/{target_name.replace('-', '_')}_routing_analysis.csv",
        index=False,
    )


## Interpretation guide

### Best-case result

A selected λ that:

- beats the static 2:1 baseline on validation RMSE / R²,
- remains non-collapsed (meaningful NN weight),
- and then also beats the static baseline on the untouched test set.

That would give us a strong R2 story.

### If validation improves but test does not

Do **not** retune λ on the test set. Report the validation-selected λ and the resulting test performance honestly.

### If no λ beats static 2:1

That's still publishable evidence: the sweep can demonstrate that entropy regularization controls routing collapse, while adaptive routing does not necessarily improve predictive accuracy over a strong fixed expert weighting baseline.
